# Referral Copilot — Data Profiling Notebook

**Track 3: Referral Copilot** — *Where should a patient or coordinator go?*

This notebook profiles the three source tables and answers the only question that
matters before any modeling: **is the data trustworthy enough to route a patient on?**

It produces a reproducible **Data Readiness Scorecard** covering:

1. Shape, schema & duplicates
2. Structured-field integrity (and CSV column-shift corruption)
3. Geographic validity (coordinates + pincodes)
4. The free-text capability fields that carry the actual referral signal
5. **The join chain** `facilities → district → NFHS need context` — the make-or-break path
6. NFHS completeness (the demand-side layer)

Tables expected in the catalog/schema:
`facilities`, `india_post_pincode_directory`, `nfhs_5_district_health_indicators`.

> Every metric is computed live so this reruns cleanly on the full 10k dataset in Databricks.


## 0. Config & setup

`spark` is provided by the Databricks runtime. If the three tables are not yet
registered, uncomment the bootstrap block to register them from CSV/volume paths.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import DataFrame

# --- table names (edit catalog/schema prefix if needed, e.g. "main.health.facilities") ---
T_FAC  = "databricks_virtue_foundation_dataset_dais_2026.virtue_foundation_dataset.facilities"
T_PIN  = "databricks_virtue_foundation_dataset_dais_2026.virtue_foundation_dataset.india_post_pincode_directory"
T_NFHS = "databricks_virtue_foundation_dataset_dais_2026.virtue_foundation_dataset.nfhs_5_district_health_indicators"

# District alias crosswalk that fixes Join 2 (workspace file location)
ALIAS_PATH = "/Workspace/Users/shloksheth.13@gmail.com/district_alias_crosswalk.csv"

# India bounding box (lat/lon) used for coordinate sanity checks
LAT_MIN, LAT_MAX = 6.0, 37.5
LON_MIN, LON_MAX = 68.0, 97.5

# Expected controlled vocabularies for the structured facility fields
GOOD_FACILITY_TYPES = ["hospital", "clinic", "dentist", "doctor",
                       "pharmacy", "farmacy", "nursing_home"]
GOOD_OPERATOR_TYPES = ["private", "public", "government"]

# Accumulates the final scorecard
readiness = {}

# --- OPTIONAL bootstrap: register tables from files if they do not exist yet ---
# base = "/Volumes/main/health/raw"   # or "dbfs:/FileStore/referral"
# (spark.read.option("header", True).option("multiLine", True).option("escape", '"')
#       .csv(f"{base}/facilities.csv").write.mode("overwrite").saveAsTable(T_FAC))
# (spark.read.option("header", True)
#       .csv(f"{base}/india_post_pincode_directory.csv").write.mode("overwrite").saveAsTable(T_PIN))
# (spark.read.option("header", True)
#       .csv(f"{base}/nfhs_5_district_health_indicators.csv").write.mode("overwrite").saveAsTable(T_NFHS))


## 1. Load & shape

Row counts and column counts. `facilities` is the messy 10k-record supply table;
`india_post_pincode_directory` is the geographic spine; `nfhs_5_*` is the
district-level demand/need layer.

In [0]:
fac  = spark.table(T_FAC)
pin   = spark.table(T_PIN)
nfhs  = spark.table(T_NFHS)

for name, df in [("facilities", fac), ("india_post_pincode_directory", pin),
                 ("nfhs_5_district_health_indicators", nfhs)]:
    n = df.count()
    print(f"{name:38s} rows={n:>8,d}  cols={len(df.columns)}")
    readiness[f"{name}_rows"] = n
    readiness[f"{name}_cols"] = len(df.columns)


facilities                             rows=  10,088  cols=51
india_post_pincode_directory           rows= 165,627  cols=11
nfhs_5_district_health_indicators      rows=     706  cols=109


## 2. Schema inventory

Quick look at column names/types. The referral-relevant columns in `facilities`
are: `facilityTypeId`, `operatorTypeId`, `address_*`, `latitude`/`longitude`,
and the four free-text fields `specialties`, `procedure`, `equipment`, `capability`.

In [0]:
print("=== facilities schema ===")
fac.printSchema()
print("\n=== india_post_pincode_directory schema ===")
pin.printSchema()
print("\n=== nfhs_5 (first 12 cols) ===")
for f in nfhs.schema.fields[:12]:
    print(f"  {f.name}: {f.dataType.simpleString()}")
print(f"  ... and {len(nfhs.columns)-12} more indicator columns")


=== facilities schema ===
root
 |-- unique_id: string (nullable = true)
 |-- source_types: string (nullable = true)
 |-- source_ids: string (nullable = true)
 |-- source_content_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- organization_type: string (nullable = true)
 |-- content_table_id: string (nullable = true)
 |-- phone_numbers: string (nullable = true)
 |-- officialPhone: string (nullable = true)
 |-- email: string (nullable = true)
 |-- websites: string (nullable = true)
 |-- officialWebsite: string (nullable = true)
 |-- yearEstablished: string (nullable = true)
 |-- acceptsVolunteers: string (nullable = true)
 |-- facebookLink: string (nullable = true)
 |-- address_line1: string (nullable = true)
 |-- address_line2: string (nullable = true)
 |-- address_line3: string (nullable = true)
 |-- address_city: string (nullable = true)
 |-- address_stateOrRegion: string (nullable = true)
 |-- address_zipOrPostcode: string (nullable = true)
 |-- address_country:

## 3. `facilities` missingness profile

Percent NULL per column, worst first. Expect heavy missingness in non-referral
metadata (`acceptsVolunteers`, `area`, `capacity`, `numberDoctors`) and low
missingness on the core identity, location, and capability fields.

In [0]:
n = fac.count()
null_exprs = [(F.sum(F.col(c).isNull().cast("int")) / F.lit(n) * 100).alias(c)
              for c in fac.columns]
miss_row = fac.agg(*null_exprs).collect()[0].asDict()
miss = sorted(miss_row.items(), key=lambda kv: kv[1], reverse=True)

print(f"{'% NULL':>7}  column")
print("-" * 50)
for col, pct in miss:
    print(f"{pct:7.1f}  {col}")

# columns we should NOT rely on for matching (>50% missing)
readiness["facilities_unreliable_cols"] = [c for c, p in miss if p > 50]


 % NULL  column
--------------------------------------------------
    1.2  latitude
    1.2  longitude
    1.2  cluster_id
    1.2  source_urls
    1.2  number_of_facts_about_the_organization
    1.2  post_metrics_most_recent_social_media_post_date
    1.2  post_metrics_post_count
    1.2  engagement_metrics_n_followers
    1.2  engagement_metrics_n_likes
    1.2  engagement_metrics_n_engagements
    1.2  source
    1.2  coordinates
    1.1  distinct_social_media_presence_count
    1.1  affiliated_staff_presence
    1.1  custom_logo_presence
    1.1  specialties
    1.1  procedure
    1.1  equipment
    1.1  capability
    1.1  recency_of_page_update
    1.1  capacity
    1.1  numberDoctors
    1.1  area
    0.8  description
    0.8  affiliationTypeIds
    0.7  operatorTypeId
    0.7  facilityTypeId
    0.6  websites
    0.6  officialWebsite
    0.6  yearEstablished
    0.6  acceptsVolunteers
    0.6  facebookLink
    0.6  address_line1
    0.6  address_line2
    0.6  address_line3
  

## 4. Structured-field integrity & column-shift corruption

A handful of rows suffer CSV **column shift** — free-text with stray commas/quotes
pushes later values (lat/long, URLs, UUIDs, specialty arrays) into the wrong column.
We detect it by flagging `facilityTypeId` / `operatorTypeId` values that fall outside
their controlled vocabulary. In the sample this was tiny (~0.2% / ~0.1%) — the long
tail in a naive `value_counts` is misleading.

In [0]:
print("=== facilityTypeId (top values) ===")
fac.groupBy("facilityTypeId").count().orderBy(F.desc("count")).show(8, truncate=40)

print("=== operatorTypeId (top values) ===")
fac.groupBy("operatorTypeId").count().orderBy(F.desc("count")).show(8, truncate=40)

bad_ft = fac.filter(F.col("facilityTypeId").isNotNull() &
                    ~F.col("facilityTypeId").isin(GOOD_FACILITY_TYPES)).count()
bad_op = fac.filter(F.col("operatorTypeId").isNotNull() &
                    ~F.col("operatorTypeId").isin(GOOD_OPERATOR_TYPES)).count()
print(f"\nfacilityTypeId corrupted (off-vocabulary): {bad_ft}  ({bad_ft/n*100:.2f}%)")
print(f"operatorTypeId corrupted (off-vocabulary): {bad_op}  ({bad_op/n*100:.2f}%)")
readiness["facilityType_corrupt_pct"] = round(bad_ft / n * 100, 2)
readiness["operatorType_corrupt_pct"] = round(bad_op / n * 100, 2)

# public vs private mix — relevant if the copilot should surface affordable options
print("\n=== public vs private mix ===")
fac.groupBy("operatorTypeId").count().orderBy(F.desc("count")).show(5, truncate=20)


=== facilityTypeId (top values) ===
+--------------+-----+
|facilityTypeId|count|
+--------------+-----+
|      hospital| 5637|
|        clinic| 3782|
|       dentist|  490|
|          NULL|   67|
|          null|   59|
|        doctor|   21|
|       farmacy|   10|
|      pharmacy|    2|
+--------------+-----+
only showing top 8 rows
=== operatorTypeId (top values) ===
+----------------------------------------+-----+
|                          operatorTypeId|count|
+----------------------------------------+-----+
|                                 private| 8842|
|                                    null|  688|
|                                  public|  469|
|                                    NULL|   73|
|                              government|    2|
|["https://www.justdial.com/Hissar/Gyn...|    1|
|                      26.885332107543945|    1|
|["https://www.justdial.com/Pathankot/...|    1|
+----------------------------------------+-----+
only showing top 8 rows

facilityTypeId 

## 5. Geographic validity (coordinates)

Coordinates are the most robust location key (and the fallback when a pincode is
missing). We coerce to double and check they land inside India's bounding box.

In [0]:
lat = F.col("latitude").cast("double")
lon = F.col("longitude").cast("double")
in_india = lat.between(LAT_MIN, LAT_MAX) & lon.between(LON_MIN, LON_MAX)

geo = fac.select(
    F.sum(in_india.cast("int")).alias("in_india"),
    F.sum(F.col("latitude").isNull().cast("int")).alias("lat_null"),
).collect()[0]

print(f"coordinates inside India bbox: {geo['in_india']:,} ({geo['in_india']/n*100:.1f}%)")
print(f"latitude NULL:                 {geo['lat_null']:,} ({geo['lat_null']/n*100:.1f}%)")
readiness["coords_valid_pct"] = round(geo["in_india"] / n * 100, 1)


coordinates inside India bbox: 9,964 (98.8%)
latitude NULL:                 118 (1.2%)


## 6. Pincode validity

The pincode is the primary join key to the postal directory. We extract a clean
6-digit code from `address_zipOrPostcode`.

In [0]:
pin6 = F.regexp_extract(F.col("address_zipOrPostcode").cast("string"), r"(\d{6})", 1)
fac = fac.withColumn("pincode6", F.when(pin6 != "", pin6).otherwise(F.lit(None)))

valid_pin = fac.filter(F.col("pincode6").isNotNull()).count()
print(f"facilities with a valid 6-digit pincode: {valid_pin:,} ({valid_pin/n*100:.1f}%)")
readiness["pincode_valid_pct"] = round(valid_pin / n * 100, 1)


facilities with a valid 6-digit pincode: 9,791 (97.1%)


## 7. Free-text capability fields — the referral signal

`specialties`, `procedure`, `equipment`, `capability` carry the actual
"can this facility do what the patient needs" answer. They are stored as
**JSON arrays** (parseable, not free regex) but messy *inside* — duplicated
tokens and a mix of a camelCase taxonomy with plain English. This cell measures
presence and JSON-array shape; normalizing these into a controlled vocabulary is
the core extraction task for the copilot.

In [0]:
text_cols = ["specialties", "procedure", "equipment", "capability", "description"]
print(f"{'field':12s} {'present':>8s} {'present%':>9s} {'json_array%':>12s}")
print("-" * 45)
for c in text_cols:
    sub = fac.filter(F.col(c).isNotNull())
    present = sub.count()
    is_arr = sub.filter(F.trim(F.col(c)).startswith("[")).count()
    arr_pct = (is_arr / present * 100) if present else 0
    print(f"{c:12s} {present:>8,d} {present/n*100:>8.1f}% {arr_pct:>11.0f}%")

print("\n--- sample capability values ---")
(fac.select("name", "capability")
    .filter(F.length("capability") > 50)
    .show(3, truncate=90))


field         present  present%  json_array%
---------------------------------------------
specialties     9,973     98.9%         100%
procedure       9,973     98.9%         100%
equipment       9,973     98.9%          99%
capability      9,973     98.9%         100%
description    10,008     99.2%           0%

--- sample capability values ---
+-------------------------+------------------------------------------------------------------------------------------+
|                     name|                                                                                capability|
+-------------------------+------------------------------------------------------------------------------------------+
|     Aravind Eye Hospital|["Eye care hospital","Located in Hyderabad","Aravind Eye Hospital Pvt. Ltd. is listed i...|
|Fortis Hospital, Gurugram|["Houses Gurgaon’s first stroke centre","Neurology department led by Dr. Praveen Gupta ...|
|Fortis Hospital Anandapur|["Specialist cancer hospital

## 8. JOIN 1 — facility → district (via pincode)

The first link in the chain. We build a `pincode → district` lookup from the
postal directory and left-join facilities onto it.

In [0]:
pin_clean = pin.withColumn(
    "pincode6",
    F.regexp_extract(F.col("pincode").cast("string"), r"(\d{6})", 1))
pin_lookup = (pin_clean.filter(F.col("pincode6") != "")
              .groupBy("pincode6")
              .agg(F.first("district").alias("district"),
                   F.first("statename").alias("statename")))

fac_dist = fac.join(pin_lookup, on="pincode6", how="left")
matched = fac_dist.filter(F.col("district").isNotNull()).count()
print(f"facilities resolved to a district via pincode: {matched:,} ({matched/n*100:.1f}%)")
readiness["join1_facility_to_district_pct"] = round(matched / n * 100, 1)


facilities resolved to a district via pincode: 9,572 (94.9%)


## 8b. Load the district alias crosswalk

Join 2 breaks on ~13% of NFHS districts due to **renames and transliteration
variants** (Allahabad→Prayagraj, Bangalore→Bengaluru, Belgaum→Belagavi). We load a
curated 90-row crosswalk from the workspace that maps every unmatched NFHS district
to its exact postal-directory spelling. It is a small lookup, so we read it with
pandas and lift it into Spark.

In [0]:
import pandas as pd

# Small lookup file living in the workspace -> read with pandas, lift to Spark
xwalk_pd = pd.read_csv(ALIAS_PATH)
xwalk = spark.createDataFrame(xwalk_pd)

print(f"alias crosswalk rows: {xwalk.count()}")
print("match types:")
xwalk.groupBy("match_type").count().orderBy(F.desc("count")).show(truncate=False)
xwalk.select("nfhs_district", "directory_district", "match_type").show(8, truncate=False)
readiness["alias_crosswalk_rows"] = xwalk.count()

# --- Spark-native alternative for larger lookups (workspace files are on the driver FS):
# xwalk = spark.read.option("header", True).csv(f"file:{ALIAS_PATH}")


alias crosswalk rows: 90
match types:
+------------------+-----+
|match_type        |count|
+------------------+-----+
|transliteration   |49   |
|official_rename   |23   |
|directional/format|16   |
|approximate       |2    |
+------------------+-----+

+---------------+------------------+---------------+
|nfhs_district  |directory_district|match_type     |
+---------------+------------------+---------------+
|Ahmadnagar     |AHMEDNAGAR        |transliteration|
|Allahabad      |PRAYAGRAJ         |official_rename|
|Aravali        |ARVALLI           |transliteration|
|Badgam         |Budgam            |transliteration|
|Bandipore      |BANDIPORA         |transliteration|
|Bangalore      |BENGALURU URBAN   |official_rename|
|Bangalore Rural|BENGALURU RURAL   |official_rename|
|Baramula       |BARAMULLA         |transliteration|
+---------------+------------------+---------------+
only showing top 8 rows


In [0]:
# Diagnostic: Check district name formats
print("=== Sample NFHS districts (first 10, alphabetical) ===" )
display(nfhs.select("district_name").distinct().orderBy("district_name").limit(10).toPandas())

print("\n=== Check if specific districts exist in NFHS ===" )
for name in ["Ahmadnagar", "Allahabad", "Bangalore", "Ahmednagar"]:
    count = nfhs.filter(F.col("district_name") == name).count()
    print(f"'{name}': {count} rows")

print("\n=== Crosswalk entries for these districts ===" )
display(xwalk.filter(F.col("nfhs_district").isin(["Ahmadnagar", "Allahabad", "Bangalore", "Ahmednagar"])).toPandas())

=== Sample NFHS districts (first 10, alphabetical) ===


district_name
Adilabad
Agar Malwa
Agra
Ahmadabad
Ahmadnagar
Aizawl
Ajmer
Akola
Alappuzha
Aligarh



=== Check if specific districts exist in NFHS ===
'Ahmadnagar': 0 rows
'Allahabad': 0 rows
'Bangalore': 0 rows
'Ahmednagar': 0 rows

=== Crosswalk entries for these districts ===


nfhs_district,directory_district,state_ut,match_type,target_in_directory
Ahmadnagar,AHMEDNAGAR,null,transliteration,true
Allahabad,PRAYAGRAJ,null,official_rename,true
Bangalore,BENGALURU URBAN,null,official_rename,true


## 9. JOIN 2 — district → NFHS need context  ✅ fixed via crosswalk

We remap NFHS district names through the crosswalk, fall back to the original name
when no alias applies, then normalize and match. This lifts the district match from
~87% to ~100% and closes the weak link in the chain.

In [0]:
def norm_district(c):
    return F.regexp_replace(F.lower(F.trim(F.col(c))), "[^a-z]", "")

# Remap NFHS names through the crosswalk; keep original where no alias exists
# IMPORTANT: Trima both sides because NFHS has trailing spaces!
nfhs_mapped = (nfhs.join(xwalk, 
                         F.trim(nfhs.district_name) == F.trim(xwalk.nfhs_district), 
                         "left")
                   .withColumn("district_resolved",
                               F.coalesce(F.col("directory_district"),
                                          F.trim(F.col("district_name")))))

nfhs_d_raw = nfhs.select(norm_district("district_name").alias("d")).distinct()
nfhs_d     = nfhs_mapped.select(norm_district("district_resolved").alias("d")).distinct()
pin_d      = pin.select(norm_district("district").alias("d")).distinct()

n_nfhs = nfhs_d_raw.count()
before = nfhs_d_raw.join(pin_d, on="d", how="inner").count()
after  = nfhs_d.join(pin_d, on="d", how="inner").count()
still_unmatched = nfhs_d.join(pin_d, on="d", how="left_anti")

print(f"NFHS districts:               {n_nfhs}")
print(f"matched BEFORE crosswalk:     {before} ({before/n_nfhs*100:.1f}%)")
print(f"matched AFTER  crosswalk:     {after} ({after/n_nfhs*100:.1f}%)")
print(f"remaining unmatched:          {still_unmatched.count()}")
readiness["join2_before_pct"] = round(before / n_nfhs * 100, 1)
readiness["join2_nfhs_match_pct"] = round(after / n_nfhs * 100, 1)
readiness["join2_unmatched_districts"] = still_unmatched.count()

if still_unmatched.count():
    print("\n--- still unmatched (review) ---")
    still_unmatched.orderBy("d").show(20, truncate=False)


NFHS districts:               698
matched BEFORE crosswalk:     608 (87.1%)
matched AFTER  crosswalk:     697 (99.9%)
remaining unmatched:          0


## 10. NFHS completeness (demand-side layer)

The need context only helps if it is complete. In the sample, the 100+ indicator
columns were ~0% missing — effectively pristine.

In [0]:
nfhs_n = nfhs.count()
id_cols = {"district_name", "state_ut", "households_surveyed",
           "women_15_49_interviewed", "men_15_54_interviewed"}
ind_cols = [c for c in nfhs.columns if c not in id_cols]

exprs = [(F.sum(F.col(c).isNull().cast("int")) / F.lit(nfhs_n) * 100).alias(c)
         for c in ind_cols]
row = nfhs.agg(*exprs).collect()[0].asDict()
avg_miss = sum(row.values()) / len(row)
worst = sorted(row.items(), key=lambda kv: kv[1], reverse=True)[:5]

print(f"NFHS districts: {nfhs_n} | states/UTs: {nfhs.select('state_ut').distinct().count()}")
print(f"avg missingness across {len(ind_cols)} indicators: {avg_miss:.2f}%")
print("worst-covered indicators:", [(c, round(p,1)) for c, p in worst])
readiness["nfhs_avg_missing_pct"] = round(avg_miss, 2)


NFHS districts: 706 | states/UTs: 36
avg missingness across 104 indicators: 0.00%
worst-covered indicators: [('female_population_age_6_years_and_above_ever_schooled_pct', 0.0), ('population_below_age_15_years_pct', 0.0), ('sex_ratio_total_f_per_1000_m', 0.0), ('sex_ratio_at_birth_5y_f_per_1000_m', 0.0), ('child_u5_whose_birth_was_civil_reg_pct', 0.0)]


## 11. Duplicate / dedup signal

A referral list must not show the same facility three times. We check exact-name
duplicates and the pre-computed `cluster_id` grouping.

In [0]:
dup_names = fac.groupBy("name").count().filter(F.col("count") > 1)
n_dup_name_rows = dup_names.agg(F.sum("count")).collect()[0][0] or 0
n_dup_name_keys = dup_names.count()

clusters = fac.filter(F.col("cluster_id").isNotNull())
n_clustered = clusters.count()
n_cluster_ids = clusters.select("cluster_id").distinct().count()

print(f"distinct names appearing >1x:        {n_dup_name_keys}")
print(f"rows involved in name duplicates:    {n_dup_name_rows}")
print(f"rows with cluster_id:                {n_clustered:,} -> {n_cluster_ids:,} clusters")
print("(near-duplicates from multiple sources likely collapse via cluster_id)")
readiness["duplicate_name_keys"] = n_dup_name_keys


distinct names appearing >1x:        321
rows involved in name duplicates:    877
rows with cluster_id:                9,970 -> 9,959 clusters
(near-duplicates from multiple sources likely collapse via cluster_id)


## 12. Trust & verification score  (genuineness — item 1)

A capability claim is only useful if the record is *real and current*. This section
combines the corroboration metadata into a per-facility **verification score (0–100)**
and tiers facilities High / Medium / Low. Components: independent-source corroboration
(`source_ids`, `source_types`), named staff, custom logo, social presence, and
reachable contact channel. We also flag impossible **future-dated** page updates as a
data-quality issue.

In [0]:
# --- robust distinct-count over JSON-array string columns ---
def distinct_count(colname):
    expr = (f"size(array_distinct(filter(from_json({colname}, 'array<string>'), "
            f"x -> x is not null and x != '')))")
    return F.greatest(F.expr(expr), F.lit(0))

fac = (fac
    .withColumn("src_id_distinct",   distinct_count("source_ids"))
    .withColumn("src_type_distinct", distinct_count("source_types"))
    .withColumn("has_staff",  (F.col("affiliated_staff_presence") == "true").cast("int"))
    .withColumn("has_logo",   (F.col("custom_logo_presence") == "true").cast("int"))
    .withColumn("has_social", (F.expr("try_cast(distinct_social_media_presence_count as double)") > 0).cast("int"))
    .withColumn("has_contact", (
        F.col("officialPhone").isNotNull() | F.col("phone_numbers").isNotNull() |
        F.col("email").isNotNull() | F.col("officialWebsite").isNotNull() |
        F.col("websites").isNotNull()).cast("int"))
    .withColumn("page_date", F.expr("try_cast(recency_of_page_update as date)")))

# verification score (0-100): weighted, documented components
fac = fac.withColumn("verification_score",
    (F.least(F.col("src_id_distinct"), F.lit(5)) / 5 * 25)      # multi-source corroboration
    + F.when(F.col("src_type_distinct") >= 2, 15).otherwise(0)  # independent source types
    + F.col("has_staff")  * 20                                  # named affiliated staff
    + F.col("has_logo")   * 10                                  # branded/owned web presence
    + F.col("has_social") * 10                                  # social footprint
    + F.col("has_contact")* 20)                                 # reachable

fac = fac.withColumn("trust_tier",
    F.when(F.col("verification_score") >= 70, "High")
     .when(F.col("verification_score") >= 40, "Medium")
     .otherwise("Low"))

future_dated = fac.filter(F.col("page_date") > F.current_date()).count()
print("verification score distribution:")
fac.selectExpr("round(percentile_approx(verification_score, 0.1),0) p10",
               "round(percentile_approx(verification_score, 0.5),0) median",
               "round(percentile_approx(verification_score, 0.9),0) p90").show()
print("trust tiers:")
fac.groupBy("trust_tier").count().orderBy(F.desc("count")).show()
print(f"future-dated page updates (data-quality flag): {future_dated}")

med = fac.selectExpr("percentile_approx(verification_score,0.5)").collect()[0][0]
high = fac.filter(F.col("trust_tier") == "High").count()
readiness["verification_score_median"] = round(float(med), 0)
readiness["high_trust_pct"] = round(high / n * 100, 1)
readiness["future_dated_pages"] = future_dated


verification score distribution:
+----+------+-----+
| p10|median|  p90|
+----+------+-----+
|55.0|  85.0|100.0|
+----+------+-----+

trust tiers:
+----------+-----+
|trust_tier|count|
+----------+-----+
|      High| 7317|
|    Medium| 2463|
|       Low|  308|
+----------+-----+

future-dated page updates (data-quality flag): 1


## 13. Claim-inflation detection  (genuineness — item 2)

Specialty arrays are capped near 50, and median claims scale with facility size
(hospital ~39, clinic ~15, solo doctor ~7). A *non-hospital* listing a hospital-sized
menu of specialties is almost certainly aggregated marketing, not genuine capacity —
the kind of place that should not receive a complex referral. We profile claim counts
by type and flag implausible breadth, including specialties-per-doctor where known.

In [0]:
fac = (fac
    .withColumn("spec_count", F.greatest(
        F.expr("size(from_json(specialties, 'array<string>'))"), F.lit(0)))
    .withColumn("n_doctors", F.expr("try_cast(numberDoctors as double)")))

print("claim count distribution by facility type:")
(fac.filter(F.col("facilityTypeId").isin("hospital", "clinic", "dentist", "doctor"))
    .groupBy("facilityTypeId")
    .agg(F.expr("percentile_approx(spec_count, 0.5)").alias("median_claims"),
         F.expr("percentile_approx(spec_count, 0.9)").alias("p90_claims"),
         F.max("spec_count").alias("max_claims"))
    .orderBy("facilityTypeId").show())

# Flag 1: non-hospital claiming hospital-scale breadth
inflated_breadth = ((F.col("facilityTypeId").isin("clinic", "dentist", "doctor")) &
                    (F.col("spec_count") >= 25))
# Flag 2: implausible specialties-per-doctor (only where doctor count is known & sane)
specs_per_doc = F.col("spec_count") / F.col("n_doctors")
inflated_ratio = (F.col("n_doctors").between(1, 1000) & (specs_per_doc > 15))

fac = fac.withColumn("claim_inflation_flag",
                     (inflated_breadth | inflated_ratio).cast("int"))
flagged = fac.filter(F.col("claim_inflation_flag") == 1).count()
print(f"facilities flagged for claim inflation: {flagged} ({flagged/n*100:.1f}%)")
print("\nsample flagged (non-hospital, very broad claims):")
(fac.filter(inflated_breadth)
    .select("name", "facilityTypeId", "spec_count", "n_doctors")
    .orderBy(F.desc("spec_count")).show(8, truncate=40))
readiness["claim_inflation_flagged_pct"] = round(flagged / n * 100, 1)


claim count distribution by facility type:
+--------------+-------------+----------+----------+
|facilityTypeId|median_claims|p90_claims|max_claims|
+--------------+-------------+----------+----------+
|        clinic|           15|        50|        50|
|       dentist|           13|        50|        50|
|        doctor|            7|        26|        50|
|      hospital|           39|        50|        50|
+--------------+-------------+----------+----------+

facilities flagged for claim inflation: 2152 (21.3%)

sample flagged (non-hospital, very broad claims):
+----------------------------------------+--------------+----------+---------+
|                                    name|facilityTypeId|spec_count|n_doctors|
+----------------------------------------+--------------+----------+---------+
|                        Belle Vue Clinic|        clinic|        50|      1.0|
|                    S.S. Dental Hospital|        clinic|        50|      2.0|
|     HCG Aastha Cancer Centre, A

## 14. Internal-consistency corroboration  (genuineness — item 3)

Genuine capability leaves fingerprints across fields: a facility that truly does
cardiology should show cardiac signals in `procedure` / `equipment` / `capability`,
not just list "cardiology" under specialties. For each high-stakes specialty we measure
the **corroboration rate** — of facilities claiming it, how many show supporting
evidence in the free text. Low rates mark specialties where the claim is often
unsupported and should carry a "verify" caveat in the copilot.

In [0]:
# specialty -> corroborating keyword roots expected in the free-text fields
COR = {
 "cardiology":       ["cardi", "ecg", "echo", "angio"],
 "radiology":        ["x-ray", "xray", "mri", "ultrasound", "sonograph", "radiolog", "scan"],
 "nephrology":       ["dialysis", "nephro", "kidney"],
 "orthopedic":       ["ortho", "fracture", "joint", "arthro"],
 "gynecology":       ["obstetr", "gyne", "delivery", "maternity", "cesarean", "csection"],
 "oncology":         ["oncolog", "chemo", "cancer", "radiation", "tumor"],
 "ophthalmology":    ["ophthalm", "cataract", "retina", "lasik"],
 "neurology":        ["neuro", "stroke", "eeg", "epilep"],
 "gastroenterology": ["gastro", "endoscop", "colonoscop", "liver"],
}

# combined lowercased evidence text + lowercased specialty string
evtxt = F.lower(F.concat_ws(" ",
    F.coalesce(F.col("procedure"),  F.lit("")),
    F.coalesce(F.col("equipment"),  F.lit("")),
    F.coalesce(F.col("capability"), F.lit(""))))
fac = (fac.withColumn("_evtxt", evtxt)
          .withColumn("_spec_lc", F.lower(F.coalesce(F.col("specialties"), F.lit("")))))

# one pass: for each specialty, count claimed and corroborated
agg_exprs = []
for spec, kws in COR.items():
    claimed = F.col("_spec_lc").contains(spec)
    evidence = None
    for k in kws:
        cond = F.col("_evtxt").contains(k)
        evidence = cond if evidence is None else (evidence | cond)
    agg_exprs.append(F.sum(claimed.cast("int")).alias(f"{spec}__claimed"))
    agg_exprs.append(F.sum((claimed & evidence).cast("int")).alias(f"{spec}__corrob"))

row = fac.agg(*agg_exprs).collect()[0].asDict()
print(f"{'specialty':16s} {'claimed_by':>10s} {'corroborated':>13s}")
print("-" * 42)
rates = []
for spec in COR:
    c, k = row[f"{spec}__claimed"], row[f"{spec}__corrob"]
    pct = (k / c * 100) if c else 0
    rates.append(pct)
    print(f"{spec:16s} {c:>10d} {pct:>12.1f}%")
readiness["avg_specialty_corroboration_pct"] = round(sum(rates) / len(rates), 1)

fac = fac.drop("_evtxt", "_spec_lc")


specialty        claimed_by  corroborated
------------------------------------------
cardiology             3085         74.2%
radiology              3200         81.9%
nephrology             2199         69.1%
orthopedic             3622         75.4%
gynecology             4601         66.1%
oncology               2208         68.3%
ophthalmology          2868         54.2%
neurology              2060         71.4%
gastroenterology       2481         70.5%


## 15. Data Readiness Scorecard

Single-glance summary of everything above — the numbers a planner needs to decide
how far to trust an automated referral, and what to fix first.

In [0]:
import pandas as pd

score = pd.DataFrame(
    [(k, v) for k, v in readiness.items()
     if not isinstance(v, list)],
    columns=["metric", "value"])
display(score)

print("\nKEY TAKEAWAYS")
print("-" * 60)
print(f"Supply rows:            {readiness.get('facilities_rows'):,}")
print(f"Valid coordinates:      {readiness.get('coords_valid_pct')}%")
print(f"Valid pincodes:         {readiness.get('pincode_valid_pct')}%")
print(f"JOIN1 facility->dist:   {readiness.get('join1_facility_to_district_pct')}%")
print(f"JOIN2 dist->NFHS:       {readiness.get('join2_before_pct')}% -> "
      f"{readiness.get('join2_nfhs_match_pct')}% after crosswalk "
      f"({readiness.get('join2_unmatched_districts')} still unmatched)")
print(f"NFHS missingness:       {readiness.get('nfhs_avg_missing_pct')}%")
print(f"Struct-field corruption: facilityType {readiness.get('facilityType_corrupt_pct')}%  "
      f"operatorType {readiness.get('operatorType_corrupt_pct')}%")
print("\n-- GENUINENESS SIGNALS --")
print(f"Median verification score:  {readiness.get('verification_score_median')}/100")
print(f"High-trust facilities:      {readiness.get('high_trust_pct')}%")
print(f"Claim-inflation flagged:    {readiness.get('claim_inflation_flagged_pct')}%")
print(f"Avg specialty corroboration:{readiness.get('avg_specialty_corroboration_pct')}%")
print(f"Future-dated pages (bad):   {readiness.get('future_dated_pages')}")
print("\nJOIN CHAIN now resolves end-to-end for ~94% of facilities.")
print("NEXT: parse the 4 JSON capability fields into a controlled vocabulary,")
print("      then combine capability + verification_score into the ranked referral table.")


metric,value
facilities_rows,10088.0
facilities_cols,51.0
india_post_pincode_directory_rows,165627.0
india_post_pincode_directory_cols,11.0
nfhs_5_district_health_indicators_rows,706.0
nfhs_5_district_health_indicators_cols,109.0
facilityType_corrupt_pct,0.77
operatorType_corrupt_pct,6.96
coords_valid_pct,98.8
pincode_valid_pct,97.1



KEY TAKEAWAYS
------------------------------------------------------------
Supply rows:            10,088
Valid coordinates:      98.8%
Valid pincodes:         97.1%
JOIN1 facility->dist:   94.9%
JOIN2 dist->NFHS:       87.1% -> 99.9% after crosswalk (0 still unmatched)
NFHS missingness:       0.0%
Struct-field corruption: facilityType 0.77%  operatorType 6.96%

-- GENUINENESS SIGNALS --
Median verification score:  85.0/100
High-trust facilities:      72.5%
Claim-inflation flagged:    21.3%
Avg specialty corroboration:70.1%
Future-dated pages (bad):   1

JOIN CHAIN now resolves end-to-end for ~94% of facilities.
NEXT: parse the 4 JSON capability fields into a controlled vocabulary,
      then combine capability + verification_score into the ranked referral table.


## 15. Capability extraction → controlled vocabulary

This is the build that powers matching: collapse the messy free-text into a clean,
queryable **facility × specialty** table. We map the camelCase `specialties` taxonomy
(2,900+ raw tokens) onto ~33 canonical categories, and corroborate each against the
free-text `procedure` / `equipment` / `capability` fields. Every (facility, specialty)
row carries a **confidence**:

- **Confirmed** — claimed in `specialties` *and* backed by procedure/equipment/capability text
- **Claimed** — listed under specialties only (no supporting evidence found)
- **Inferred** — appears only in free text, not in the specialties list (a lead to verify)

Validated coverage: ~99% of specialty-token occurrences map, and ~99% of facilities get
at least one canonical specialty (median ~6). The output view `facility_capabilities`
carries each facility's verification score and trust tier so the copilot can rank on
*capability × genuineness* in one place.

In [0]:
import json, re
from pyspark.sql.types import (ArrayType, StructType, StructField,
                               StringType, BooleanType)

# Canonical ontology: category -> spaced lowercase roots (matched on de-camelCased text)
ONTOLOGY = {
 "Internal Medicine":["internal medicine","general medicine","geriatric","hospice","palliative","sleep medicine","chronic disease"],
 "Family Medicine":["family medicine","general practice"],
 "Preventive & Public Health":["preventive medicine","public health","lifestyle medicine","community medicine"],
 "Cardiology":["cardiology","cardiac","ecg","echocardio","angiograph","angioplasty","stent","catheter"],
 "Cardiac Surgery":["cardiac surgery","cardiothoracic","cabg","bypass graft","valve replacement"],
 "Obstetrics & Gynecology":["gynecolog","obstetric","maternity","maternal","reproductive endocrin","perinatolog","contracept","family planning","delivery","cesarean","caesarean"],
 "Pediatrics":["pediatric","paediatric","neonatolog","newborn","pedodontic"],
 "Ophthalmology":["ophthalmolog","retina","glaucoma","cornea","cataract","oculoplast","refractive surgery","lasik","vitreoretinal","strabismus","eye trauma","eye care"],
 "Orthopedics":["orthopedic","orthopaedic","arthroscop","arthroplasty","joint replacement","joint reconstruction","fracture","spine surgery","podiatry","sports medicine"],
 "Dentistry":["dentistry","dental","endodontic","orthodontic","prosthodontic","periodontic","maxillofacial","dentoalveolar","oral medicine","root canal","pedodontic"],
 "Dermatology":["dermatolog","skin","hair and nail"],
 "Gastroenterology":["gastroenterolog","endoscop","colonoscop","hepatolog","hepatopancreat"],
 "General Surgery":["general surgery","colorectal","bariatric","transplant surgery","minimally invasive surgery","laparoscop","hernia","breast surgery"],
 "Neurology":["neurology","stroke","epilep","headache","multiple sclerosis","eeg","parkinson","neurophysiolog"],
 "Neurosurgery":["neurosurgery","thrombectomy","craniotomy","deep brain stimulation"],
 "Oncology":["oncolog","cancer","chemotherap","radiotherap","radiation therapy","tumour","tumor","linear accelerator","brachytherap"],
 "Urology":["urology","endourolog","prostate","kidney stone","androl"],
 "Nephrology":["nephrolog","dialysis","renal","kidney transplant"],
 "Pulmonology":["pulmonolog","respiratory","asthma","copd","bronchoscop"],
 "ENT (Otolaryngology)":["otolaryngolog","head and neck surgery","sinus","cochlear","ear nose throat"],
 "Radiology & Imaging":["radiolog","ct scan","mri","ultrasound","sonograph","x-ray","xray","nuclear medicine","pet scan","breast imaging","mammograph","molecular imaging"],
 "Pathology & Lab":["patholog","laboratory","biopsy","histopath","microbiolog","cytolog"],
 "Psychiatry & Mental Health":["psychiatry","mental health","de-addiction","deaddiction","counsel","psycholog"],
 "Endocrinology & Diabetes":["endocrin","diabetes","thyroid","metabolism"],
 "Emergency & Critical Care":["emergency medicine","emergency care","critical care","intensive care","trauma care","casualty"],
 "Anesthesia & Pain":["anesthe","anaesthe","pain medicine","pain management"],
 "Physical Med & Rehab":["physical medicine","rehabilitation","physiotherap","occupational therap"],
 "Plastic & Reconstructive":["plastic surgery","cosmetic surgery","reconstructive","aesthetic","burn"],
 "Rheumatology":["rheumatolog","arthritis","autoimmune"],
 "Hematology":["hematolog","haematolog","blood disorder","bone marrow"],
 "Vascular Surgery":["vascular surgery","vascular"],
 "Infectious Disease":["infectious disease"],
 "Allergy & Immunology":["allergy","immunolog"],
 "AYUSH / Traditional":["ayurved","panchkarma","homeopath","unani","siddha","naturopath"],
}

def _decamel(s):
    s = re.sub(r"(?<=[a-z])(?=[A-Z])", " ", s)
    s = re.sub(r"(?<=[A-Z])(?=[A-Z][a-z])", " ", s)
    return re.sub(r"\s+", " ", s).lower().strip()

def _jtext(x):
    if not x: return ""
    try:
        v = json.loads(x)
        if isinstance(v, list): return " ".join(str(t) for t in v if t)
    except Exception:
        pass
    return str(x)

_schema = ArrayType(StructType([
    StructField("specialty",  StringType()),
    StructField("claimed",    BooleanType()),
    StructField("evidence",   BooleanType()),
    StructField("confidence", StringType()),
]))

def _extract(specialties, procedure, equipment, capability):
    spec_text = ""
    if specialties:
        try:
            v = json.loads(specialties)
            spec_text = " ".join(_decamel(t) for t in v if isinstance(t, str)) \
                        if isinstance(v, list) else _decamel(str(specialties))
        except Exception:
            spec_text = _decamel(str(specialties))
    ev_text = (_jtext(procedure) + " " + _jtext(equipment) + " " + _jtext(capability)).lower()
    out = []
    for cat, roots in ONTOLOGY.items():
        claimed  = any(r in spec_text for r in roots)
        evidence = any(r in ev_text   for r in roots)
        if claimed or evidence:
            conf = "Confirmed" if (claimed and evidence) else ("Claimed" if claimed else "Inferred")
            out.append((cat, claimed, evidence, conf))
    return out

extract_udf = F.udf(_extract, _schema)

# Explode into the long facility x specialty table, carrying trust signals
facility_capabilities = (
    fac.withColumn("_caps", extract_udf("specialties", "procedure", "equipment", "capability"))
       .select("unique_id", "name", "facilityTypeId", "operatorTypeId",
               "verification_score", "trust_tier", "pincode6",
               F.explode("_caps").alias("c"))
       .select("unique_id", "name", "facilityTypeId", "operatorTypeId",
               "verification_score", "trust_tier", "pincode6",
               F.col("c.specialty").alias("specialty"),
               F.col("c.claimed").alias("claimed"),
               F.col("c.evidence").alias("evidence"),
               F.col("c.confidence").alias("confidence")))
facility_capabilities.createOrReplaceTempView("facility_capabilities")
# Persist for the app:
facility_capabilities.write.mode("overwrite").saveAsTable("workspace.default.facility_capabilities")

print("facility_capabilities sample:")
facility_capabilities.show(8, truncate=30)


facility_capabilities sample:
+------------------------------+--------------------+--------------+--------------+------------------+----------+--------+-----------------------+-------+--------+----------+
|                     unique_id|                name|facilityTypeId|operatorTypeId|verification_score|trust_tier|pincode6|              specialty|claimed|evidence|confidence|
+------------------------------+--------------------+--------------+--------------+------------------+----------+--------+-----------------------+-------+--------+----------+
|b8a5401f-42f1-422a-8cd9-686...|Aravind Eye Hospital|      hospital|       private|             100.0|      High|  110002|      Internal Medicine|   true|   false|   Claimed|
|b8a5401f-42f1-422a-8cd9-686...|Aravind Eye Hospital|      hospital|       private|             100.0|      High|  110002|             Cardiology|  false|    true|  Inferred|
|b8a5401f-42f1-422a-8cd9-686...|Aravind Eye Hospital|      hospital|       private|            

In [0]:
# --- coverage & confidence summary ---
total_rows = facility_capabilities.count()
covered = facility_capabilities.select("unique_id").distinct().count()
print(f"capability rows: {total_rows:,}  | facilities covered: {covered:,} ({covered/n*100:.1f}%)")

print("\nconfidence breakdown:")
facility_capabilities.groupBy("confidence").count().orderBy(F.desc("count")).show()

print("top canonical specialties by facility count:")
(facility_capabilities.groupBy("specialty")
    .agg(F.countDistinct("unique_id").alias("facilities"),
         F.round(F.avg((F.col("confidence") == "Confirmed").cast("int")) * 100, 0).alias("pct_confirmed"))
    .orderBy(F.desc("facilities")).show(15, truncate=False))

confirmed = facility_capabilities.filter(F.col("confidence") == "Confirmed").count()
readiness["capability_facilities_covered_pct"] = round(covered / n * 100, 1)
readiness["capability_confirmed_share_pct"] = round(confirmed / total_rows * 100, 1)

#claimed = facility_capabilities.filter(F.col("confidence") == "Claimed").count()
#readiness["capability_claimed_share_pct"] = round(claimed / total_rows * 100, 1)

#inferred = facility_capabilities.filter(F.col("confidence") == "Inferred").count()
#readiness["capability_inferred_share_pct"] = round(inferred / total_rows * 100, 1)


capability rows: 100,697  | facilities covered: 9,959 (98.7%)

confidence breakdown:
+----------+-----+
|confidence|count|
+----------+-----+
| Confirmed|45181|
|   Claimed|41117|
|  Inferred|14399|
+----------+-----+

top canonical specialties by facility count:
+-------------------------+----------+-------------+
|specialty                |facilities|pct_confirmed|
+-------------------------+----------+-------------+
|Internal Medicine        |7058      |27.0         |
|Pediatrics               |5499      |64.0         |
|Obstetrics & Gynecology  |5221      |60.0         |
|Family Medicine          |5199      |1.0          |
|Radiology & Imaging      |4774      |55.0         |
|Dentistry                |4642      |66.0         |
|General Surgery          |4187      |59.0         |
|Orthopedics              |4107      |66.0         |
|Pathology & Lab          |3844      |50.0         |
|Urology                  |3815      |57.0         |
|Emergency & Critical Care|3692      |56.0     

## 16. Evidence-snippet extraction → citations + grade  (search item #3)

The track requires that each candidate **cite the underlying facility text** and show
**matching vs missing/suspicious evidence**. This section turns the free-text into a
quotable **citation table**: for each facility and each high-stakes capability a referral
user actually searches (dialysis, ICU, NICU, maternity, emergency, trauma, oncology,
cardiac, imaging, surgery), it pulls the *exact supporting sentence* from
`description` / `capability` / `procedure` / `equipment`, with provenance.

Each citation carries a **strength** (operational evidence from `procedure`/`equipment`
outranks a `description` mention), which rolls up to an **evidence_grade** per
(facility, capability):

- **Strong** — at least one operational citation (procedure/equipment)
- **Partial** — only mentioned in capability/description text (claim, not proof)
- *(no row)* — capability not found in this facility's text → the "missing/suspicious"
  signal the app surfaces when the capability is claimed elsewhere (e.g. specialties/name)

Outputs two views: `facility_evidence` (one row per citation) and `evidence_summary`
(one row per facility × capability with its grade).

In [0]:
import json, re
from pyspark.sql.types import ArrayType, StructType, StructField, StringType

# Operational capabilities a referral user searches (challenge-named + queryable)
EVIDENCE_CAPABILITIES = {
 "Dialysis":            ["dialysis","haemodialysis","hemodialysis","renal replacement"],
 "ICU / Critical Care": ["icu","intensive care","critical care","ventilator","high dependency"],
 "NICU / Neonatal":     ["nicu","neonatal","newborn intensive","preterm"],
 "Maternity / Delivery":["maternity","obstetric","delivery","labour room","labor room","cesarean","caesarean","birthing"],
 "Emergency":           ["emergency","casualty","accident and emergency","resuscitation"],
 "Trauma":              ["trauma","polytrauma"],
 "Oncology / Cancer":   ["oncolog","cancer","chemotherap","radiotherap","radiation therapy","tumour","tumor","linear accelerator"],
 "Cardiac / Cath Lab":  ["cath lab","catheterization","angioplasty","angiograph","stent","echocardio"],
 "Advanced Imaging":    ["ct scan","mri","pet scan","sonograph","ultrasound","mammograph"],
 "Surgery (Major OT)":  ["operation theatre","operating theatre","operation theater","laparoscop","surgical procedure"],
}
# alternation pattern per capability with word-boundary start (plain strings -> picklable)
_CAP_PATTERNS = {cap: "|".join(r"\b" + re.escape(r) for r in roots)
                 for cap, roots in EVIDENCE_CAPABILITIES.items()}
_STRENGTH = {"procedure": "strong", "equipment": "strong",
             "capability": "moderate", "description": "supporting"}

# Pre-compile all regex patterns at module level for pickling
_COMPILED_PATTERNS = {cap: re.compile(pat) for cap, pat in _CAP_PATTERNS.items()}

def _units(description, capability, procedure, equipment):
    """Quotable (field, snippet) units: prose split for description, array elems for the rest."""
    out = []
    if description and isinstance(description, str):
        for s in re.split(r"(?<=[.!?])\s+", description.strip()):
            if len(s) >= 15:
                out.append(("description", s.strip()))
    for fname, val in (("capability", capability), ("procedure", procedure), ("equipment", equipment)):
        if val:
            try:
                v = json.loads(val)
                if isinstance(v, list):
                    for el in v:
                        if isinstance(el, str) and len(el.strip()) >= 8:
                            out.append((fname, el.strip()))
            except Exception:
                pass
    return out

_ev_schema = ArrayType(StructType([
    StructField("capability",     StringType()),
    StructField("evidence_field", StringType()),
    StructField("snippet",        StringType()),
    StructField("strength",       StringType()),
]))

def _evidence(description, capability, procedure, equipment):
    us = _units(description, capability, procedure, equipment)
    out, seen = [], set()
    for cap, rx in _COMPILED_PATTERNS.items():
        for f, s in us:
            if rx.search(s.lower()):
                key = (cap, s)
                if key in seen:
                    continue
                seen.add(key)
                out.append((cap, f, s[:300], _STRENGTH[f]))
    return out

evidence_udf = F.udf(_evidence, _ev_schema)

# carry provenance + trust; guard source_urls in case the table lacks it
_carry = [c for c in ["unique_id", "name", "facilityTypeId",
                      "verification_score", "trust_tier", "pincode6"] if c in fac.columns]
ev = fac.withColumn("_ev", evidence_udf("description", "capability", "procedure", "equipment"))
if "source_urls" in fac.columns:
    ev = ev.withColumn("source_urls_arr",
                       F.expr("filter(from_json(source_urls,'array<string>'), x -> x is not null and x != '')"))
else:
    ev = ev.withColumn("source_urls_arr", F.array().cast("array<string>"))

facility_evidence = (ev
    .select(*_carry, "source_urls_arr", F.explode("_ev").alias("e"))
    .select(*_carry,
            F.col("e.capability").alias("capability"),
            F.col("e.evidence_field").alias("evidence_field"),
            F.col("e.snippet").alias("snippet"),
            F.col("e.strength").alias("strength"),
            F.slice("source_urls_arr", 1, 3).alias("source_urls")))
facility_evidence.createOrReplaceTempView("facility_evidence")
facility_evidence.write.mode("overwrite").saveAsTable("workspace.default.facility_evidence")

# roll up to one grade per facility x capability
evidence_summary = (facility_evidence.groupBy(*[c for c in _carry], "capability")
    .agg(F.count("*").alias("n_citations"),
         F.max((F.col("strength") == "strong").cast("int")).alias("has_operational"),
         F.sort_array(F.collect_set("evidence_field")).alias("fields"))
    .withColumn("evidence_grade",
                F.when(F.col("has_operational") == 1, "Strong").otherwise("Partial")))
evidence_summary.createOrReplaceTempView("evidence_summary")
evidence_summary.write.mode("overwrite").saveAsTable("workspace.default.evidence_summary")
# Persist for the app:
# facility_evidence.write.mode("overwrite").saveAsTable("facility_evidence")
# evidence_summary.write.mode("overwrite").saveAsTable("evidence_summary")

print("facility_evidence sample (quotable citations):")
facility_evidence.select("name", "capability", "evidence_field", "strength", "snippet").show(8, truncate=70)


facility_evidence sample (quotable citations):
+--------------------+----------+--------------+--------+----------------------------------------------------------------------+
|                name|capability|evidence_field|strength|                                                               snippet|
+--------------------+----------+--------------+--------+----------------------------------------------------------------------+
|Aravind Eye Hospital|  Dialysis|    capability|moderate|Dialysis unit of 20 beds under Pradhan Mantri National Dialysis Pro...|
|Aravind Eye Hospital|  Dialysis|     procedure|  strong|Dialysis treatment is provided via a 20-bed Dialysis Unit with 10 m...|
|Aravind Eye Hospital|  Dialysis|     procedure|  strong|         Hemodialysis is performed in three units as a regular service|
|Aravind Eye Hospital|  Dialysis|     procedure|  strong|                     Peritoneal dialysis is available around the clock|
|Aravind Eye Hospital|  Dialysis|     procedure|  

In [0]:
# --- coverage, grade mix, and the "missing/suspicious" signal ---
total_cites = facility_evidence.count()
fac_with_ev = facility_evidence.select("unique_id").distinct().count()
print(f"citations: {total_cites:,} | facilities with >=1 citation: {fac_with_ev:,} ({fac_with_ev/n*100:.1f}%)")

print("\nevidence grade per capability (Strong = operational proof; Partial = mention only):")
(evidence_summary.groupBy("capability")
    .pivot("evidence_grade", ["Strong", "Partial"]).count()
    .na.fill(0).orderBy(F.desc("Strong")).show(truncate=False))

# Missing/suspicious demo: facilities whose NAME implies a capability but lack STRONG evidence.
# This is the per-candidate red flag the app shows next to a weak match.
name_claims = []
for cap, roots in EVIDENCE_CAPABILITIES.items():
    pat = "(?i)" + "|".join(roots[:3])
    claimed = fac.filter(F.col("name").rlike(pat)).select("unique_id", "name").withColumn("capability", F.lit(cap))
    name_claims.append(claimed)
from functools import reduce
name_claimed = reduce(lambda a, b: a.unionByName(b), name_claims)
strong = evidence_summary.filter(F.col("evidence_grade") == "Strong").select("unique_id", "capability")
suspicious = name_claimed.join(strong, ["unique_id", "capability"], "left_anti")
print(f"\nSUSPICIOUS: facilities naming a capability but with no STRONG evidence: {suspicious.count()}")
suspicious.show(8, truncate=45)

readiness["evidence_facilities_covered_pct"] = round(fac_with_ev / n * 100, 1)
readiness["evidence_total_citations"] = total_cites


citations: 69,728 | facilities with >=1 citation: 6,609 (65.5%)

evidence grade per capability (Strong = operational proof; Partial = mention only):
+--------------------+------+-------+
|capability          |Strong|Partial|
+--------------------+------+-------+
|Surgery (Major OT)  |3037  |252    |
|Advanced Imaging    |2536  |88     |
|Oncology / Cancer   |1695  |668    |
|Maternity / Delivery|1542  |1543   |
|Cardiac / Cath Lab  |1329  |34     |
|ICU / Critical Care |1158  |1284   |
|Emergency           |985   |2450   |
|Dialysis            |889   |162    |
|Trauma              |797   |678    |
|NICU / Neonatal     |462   |512    |
+--------------------+------+-------+


SUSPICIOUS: facilities naming a capability but with no STRONG evidence: 143
+---------------------------------------------+-------------------+---------------------------------------------+
|                                    unique_id|         capability|                                         name|
+------------

## 17. Need → capability synonym layer  (search item #1)

A user types lay language — *"kidney failure"*, *"heart attack"*, *"child specialist"*,
or the challenge's own *"dialysis near Jaipur"* / *"emergency surgery near Patna"* — not
ontology keys. This layer maps free-text needs onto the exact capability names from §16
(`evidence_summary`) and specialty names from §15 (`facility_capabilities`), so a query
resolves straight into the evidence layer.

How it resolves:
- **Word-boundary phrase match** against a curated lay+clinical synonym table.
- **Longest match wins** per target, so *"kidney failure"* → Dialysis ranks above the
  generic *"kidney"* → Nephrology, and both are returned (ranked) for breadth.
- **Location words are ignored** (they don't match any need phrase), so *"near Jaipur"*
  falls through to the geocoding step (item #2) untouched.
- **Multi-need queries split**, e.g. *"emergency surgery"* → Emergency + Surgery.

Outputs `need_synonyms` (the lookup table) and a `resolve_need(query)` function the app
calls to turn a search box into ranked capability targets.

In [0]:
import re
from pyspark.sql.types import ArrayType, StructType, StructField, StringType, IntegerType

# Target keys MUST match EVIDENCE_CAPABILITIES (§16) and ONTOLOGY (§15) exactly so they join.
# target -> (target_type, [lay + clinical synonym phrases])
NEED_SYNONYMS = {
 # ---- operational capabilities (resolve -> evidence_summary) ----
 "Dialysis":              ("capability", ["dialysis","kidney failure","renal failure","kidney machine","ckd","end stage renal","esrd","kidney dialysis"]),
 "ICU / Critical Care":   ("capability", ["icu","intensive care","critical care","ventilator","life support","critical condition","ccu"]),
 "NICU / Neonatal":       ("capability", ["nicu","neonatal","newborn care","premature baby","preterm baby","sick newborn","incubator"]),
 "Maternity / Delivery":  ("capability", ["maternity","delivery","childbirth","pregnancy","labour","labor","c-section","cesarean","normal delivery","having a baby","prenatal","antenatal"]),
 "Emergency":             ("capability", ["emergency","casualty","urgent care","24 hour","24x7","24/7","acute care","ambulance"]),
 "Trauma":                ("capability", ["trauma","road accident","accident injury","broken bone","head injury","fall injury","poly trauma","fracture"]),
 "Oncology / Cancer":     ("capability", ["cancer","oncology","tumor","tumour","chemo","chemotherapy","radiation therapy","malignancy","leukemia"]),
 "Cardiac / Cath Lab":    ("capability", ["heart attack","cardiac arrest","chest pain","heart problem","cardiac","angioplasty","stent","bypass surgery","heart surgery","cath lab"]),
 "Advanced Imaging":      ("capability", ["ct scan","mri","x-ray","ultrasound","sonography","scan","imaging","mammogram","pet scan"]),
 "Surgery (Major OT)":    ("capability", ["surgery","operation","surgical","laparoscopy","operation theatre","ot procedure"]),
 # ---- specialties (resolve -> facility_capabilities) ----
 "Cardiology":            ("specialty", ["cardiology","heart doctor","heart specialist","cardiologist"]),
 "Dermatology":           ("specialty", ["skin","skin doctor","skin problem","rash","acne","dermatologist"]),
 "Ophthalmology":         ("specialty", ["eye","eye doctor","vision","cataract","eyesight","ophthalmologist"]),
 "Pediatrics":            ("specialty", ["child specialist","child doctor","pediatrician","kids doctor","baby doctor","paediatrician"]),
 "Orthopedics":           ("specialty", ["bone","joint","knee","orthopedic","back pain","joint replacement","ortho","shoulder pain","fracture"]),
 "Dentistry":             ("specialty", ["teeth","tooth","dental","dentist","root canal","braces"]),
 "ENT (Otolaryngology)":  ("specialty", ["ear","nose","throat","ent","sinus","hearing"]),
 "Gastroenterology":      ("specialty", ["stomach","gastro","liver","digestion","endoscopy","acidity"]),
 "Urology":               ("specialty", ["urology","prostate","kidney stone","bladder","urologist"]),
 "Nephrology":            ("specialty", ["kidney","nephrology","nephrologist"]),
 "Neurology":             ("specialty", ["nerve","stroke","paralysis","seizure","epilepsy","migraine","neuro","headache"]),
 "Neurosurgery":          ("specialty", ["brain surgery","neurosurgery","spine surgery","brain tumour surgery"]),
 "Psychiatry & Mental Health":("specialty", ["mental health","depression","psychiatrist","anxiety","counselling","counseling"]),
 "Pulmonology":           ("specialty", ["lung","breathing","asthma","respiratory","tb","copd","chest infection"]),
 "Endocrinology & Diabetes":("specialty", ["diabetes","sugar","thyroid","hormone","endocrine"]),
 "Obstetrics & Gynecology":("specialty", ["women's health","gynecologist","gynec","pcod","pcos","period problem","gynaecologist"]),
 "Rheumatology":          ("specialty", ["arthritis","joint pain","rheumatoid"]),
 "General Surgery":       ("specialty", ["hernia","appendix","gallbladder","gall stone","gallstone"]),
}

# flat phrase list, longest-first for specificity
_PHRASES = sorted(
    [(p, t, ttype) for t, (ttype, ps) in NEED_SYNONYMS.items() for p in ps],
    key=lambda x: -len(x[0]))

_need_schema = ArrayType(StructType([
    StructField("target",  StringType()),
    StructField("type",    StringType()),
    StructField("matched", StringType()),
    StructField("score",   IntegerType()),
]))

def resolve_need(query):
    """Free-text need -> ranked list of capability/specialty targets (location ignored)."""
    if not query:
        return []
    q = " " + re.sub(r"[^a-z0-9\s/+-]", " ", str(query).lower()) + " "
    hits = {}
    for phrase, target, ttype in _PHRASES:
        if re.search(r"(?<![a-z])" + re.escape(phrase) + r"(?![a-z])", q):
            score = len(phrase)
            if target not in hits or score > hits[target][1]:
                hits[target] = (ttype, score, phrase)
    out = [(t, v[0], v[2], v[1]) for t, v in hits.items()]
    return sorted(out, key=lambda x: -x[3])

resolve_need_udf = F.udf(resolve_need, _need_schema)

# Persisted lookup table (for the app / SQL side)
need_rows = [(p, t, ttype, len(p)) for p, t, ttype in _PHRASES]
need_synonyms = spark.createDataFrame(need_rows, ["phrase", "target", "target_type", "phrase_len"])
need_synonyms.createOrReplaceTempView("need_synonyms")
need_synonyms.write.mode("overwrite").saveAsTable("workspace.default.need_synonyms")
print(f"need_synonyms: {need_synonyms.count()} phrases -> {need_synonyms.select('target').distinct().count()} targets")


need_synonyms: 186 phrases -> 28 targets


In [0]:
# --- demo: resolve the sample queries, then check live supply via the evidence layer ---
demo = ["dialysis near Jaipur", "emergency surgery near Patna", "heart attack",
        "kidney failure", "child specialist in Pune", "MRI scan near Delhi",
        "cancer treatment", "fracture", "breathing problem"]
print("NEED RESOLUTION:")
for q in demo:
    r = resolve_need(q)
    print(f"  {q:32s} -> " + (", ".join(f"{t}[{ty}]" for t, ty, _, _ in r) if r else "[no match]"))

# End-to-end: "dialysis near Jaipur" -> Dialysis -> facilities with STRONG evidence
top = resolve_need("dialysis near Jaipur")
if top:
    target, ttype = top[0][0], top[0][1]
    print(f"\nLive supply for top target '{target}' ({ttype}):")
    if ttype == "capability":
        (evidence_summary.filter(F.col("capability") == target)
            .groupBy("evidence_grade").count().orderBy(F.desc("count")).show())
    else:
        (facility_capabilities.filter(F.col("specialty") == target)
            .groupBy("confidence").count().orderBy(F.desc("count")).show())

# Coverage QA: every synonym target should be a real capability/specialty key with supply
cap_keys = set(r["capability"] for r in evidence_summary.select("capability").distinct().collect())
spec_keys = set(r["specialty"] for r in facility_capabilities.select("specialty").distinct().collect())
orphans = [t for t, (ttype, _) in NEED_SYNONYMS.items()
           if (ttype == "capability" and t not in cap_keys) or (ttype == "specialty" and t not in spec_keys)]
print("synonym targets with NO matching supply key (should be empty):", orphans)
readiness["need_synonym_targets"] = need_synonyms.select("target").distinct().count()


NEED RESOLUTION:
  dialysis near Jaipur             -> Dialysis[capability]
  emergency surgery near Patna     -> Emergency[capability], Surgery (Major OT)[capability]
  heart attack                     -> Cardiac / Cath Lab[capability]
  kidney failure                   -> Dialysis[capability], Nephrology[specialty]
  child specialist in Pune         -> Pediatrics[specialty]
  MRI scan near Delhi              -> Advanced Imaging[capability]
  cancer treatment                 -> Oncology / Cancer[capability]
  fracture                         -> Trauma[capability], Orthopedics[specialty]
  breathing problem                -> Pulmonology[specialty]

Live supply for top target 'Dialysis' (capability):
+--------------+-----+
|evidence_grade|count|
+--------------+-----+
|        Strong|  889|
|       Partial|  162|
+--------------+-----+

synonym targets with NO matching supply key (should be empty): []


## 18. Need-aware routing via NFHS  (standout item #6)

Most teams will treat the NFHS layer as decoration. This turns it into **referral
intelligence**: for the capability a user searched, how much does the *destination
district* actually need that kind of care, and is supply there already thin?

Two ideas:
- **District need score** (0–100) per capability — built from the NFHS indicators that
  signal burden, each converted to a cross-district **percentile** and direction-adjusted
  (low institutional-birth % = high maternity need; high blood-sugar % = high cardiac/renal
  need). So "high need" is honest and relative to the rest of India.
- **Demand–supply gap** — need score vs the count of facilities with *Strong* evidence
  (§16) in that district. High need + thin verified supply = a priority gap, and a signal
  the copilot should caveat ("routing into a strained district; nearest strong option may
  be elsewhere").

NFHS values are cleaned first — some are parenthesized small-sample flags like `(27.0)`,
which are stripped before casting. Outputs `district_capability_need`,
`capability_need_indicators` (the drill-down reference), and `district_capability_gap`.

In [0]:
from pyspark.sql.window import Window

# capability/specialty target -> NFHS indicators that signal NEED for it.
# direction: "low" = a low value means high need; "high" = a high value means high need.
CAPABILITY_NEED_MAP = {
 "Maternity / Delivery": [
    ("institutional_birth_5y_pct", "low", "Institutional births %"),
    ("mothers_who_had_at_least_4_anc_visits_lb5y_pct", "low", "4+ ANC visits %"),
    ("pregnant_w15_49_who_are_anaemic_lt_11_0_g_dl_22_pct", "high", "Anaemia in pregnant women %"),
    ("births_attended_by_skilled_hp_5y_10_pct", "low", "Skilled birth attendance %")],
 "Obstetrics & Gynecology": [
    ("institutional_birth_5y_pct", "low", "Institutional births %"),
    ("mothers_who_had_at_least_4_anc_visits_lb5y_pct", "low", "4+ ANC visits %"),
    ("all_w15_49_who_are_anaemic_pct", "high", "Anaemia in women 15-49 %")],
 "NICU / Neonatal": [
    ("child_12_23m_fully_vaccinated_based_on_information_from_eit_pct", "low", "Child fully vaccinated %"),
    ("children_who_received_pnc_from_a_doctor_nurse_lhv_anm_midwi_pct", "low", "Postnatal care %"),
    ("child_u5_who_are_stunted_height_for_age_18_pct", "high", "Child stunting %")],
 "Pediatrics": [
    ("child_12_23m_fully_vaccinated_based_on_information_from_eit_pct", "low", "Child fully vaccinated %"),
    ("child_6_59m_who_are_anaemic_lt_11_0_g_dl_22_pct", "high", "Child anaemia %"),
    ("child_u5_who_are_underweight_weight_for_age_18_pct", "high", "Child underweight %")],
 "Cardiac / Cath Lab": [
    ("w15_plus_with_high_bp_sys_gte_140_mmhg_and_or_dia_gte_90_mm_pct", "high", "High BP, women %"),
    ("m15_plus_with_high_bp_sys_gte_140_mmhg_and_or_dia_gte_90_mm_pct", "high", "High BP, men %"),
    ("w15_plus_with_high_or_very_high_gt_140_mg_dl_blood_sugar_or_pct", "high", "High blood sugar, women %")],
 "Cardiology": [
    ("w15_plus_with_high_bp_sys_gte_140_mmhg_and_or_dia_gte_90_mm_pct", "high", "High BP, women %"),
    ("m15_plus_with_high_bp_sys_gte_140_mmhg_and_or_dia_gte_90_mm_pct", "high", "High BP, men %")],
 "Dialysis": [
    ("w15_plus_with_high_or_very_high_gt_140_mg_dl_blood_sugar_or_pct", "high", "High blood sugar, women %"),
    ("m15_plus_with_high_or_very_high_gt_140_mg_dl_blood_sugar_or_pct", "high", "High blood sugar, men %")],
 "Endocrinology & Diabetes": [
    ("w15_plus_with_high_or_very_high_gt_140_mg_dl_blood_sugar_or_pct", "high", "High blood sugar, women %"),
    ("m15_plus_with_high_or_very_high_gt_140_mg_dl_blood_sugar_or_pct", "high", "High blood sugar, men %")],
 "Oncology / Cancer": [
    ("women_age_30_49_years_ever_undergone_an_oral_cancer_exam_pct", "low", "Oral cancer screening %")],
}

def _norm_d(c):
    return F.regexp_replace(F.lower(F.trim(F.col(c))), "[^a-z]", "")

def _clean_num(c):  # strip "()", "*", "%", spaces -> double
    cleaned = F.regexp_replace(F.col(c).cast("string"), "[^0-9.]", "")
    return F.when(cleaned == "", None).otherwise(cleaned).cast("double")

# resolve NFHS district through the alias crosswalk (xwalk from §8b)
nfhs_res = (nfhs.join(xwalk, nfhs.district_name == xwalk.nfhs_district, "left")
                .withColumn("dist_resolved", F.coalesce("directory_district", "district_name"))
                .withColumn("dist_norm", _norm_d("dist_resolved")))

# direction-adjusted percentile (need) for every indicator we use
indicators = {ind: d for grp in CAPABILITY_NEED_MAP.values() for ind, d, _ in grp}
for ind, direction in indicators.items():
    pr = F.percent_rank().over(Window.orderBy(_clean_num(ind)))
    need = (1 - pr) if direction == "low" else pr
    nfhs_res = nfhs_res.withColumn(ind + "__need", F.round(need * 100, 0))

# per-capability need score = avg of its indicators' need percentiles -> long table
need_parts = []
for cap, inds in CAPABILITY_NEED_MAP.items():
    need_cols = [ind + "__need" for ind, _, _ in inds]
    score = sum(F.col(c) for c in need_cols) / F.lit(len(inds))
    part = (nfhs_res.withColumn("capability", F.lit(cap))
                    .withColumn("need_score", F.round(score, 0))
                    .select("dist_resolved", "dist_norm", "capability", "need_score"))
    need_parts.append(part)
from functools import reduce
district_capability_need = reduce(lambda a, b: a.unionByName(b), need_parts).withColumn(
    "need_level", F.when(F.col("need_score") >= 66, "High")
                   .when(F.col("need_score") >= 33, "Medium").otherwise("Low"))
district_capability_need.createOrReplaceTempView("district_capability_need")
district_capability_need.write.mode("overwrite").saveAsTable("workspace.default.district_capability_need")

# drill-down reference: which raw indicators explain a capability's need
ref_rows = [(cap, ind, d, label) for cap, inds in CAPABILITY_NEED_MAP.items() for ind, d, label in inds]
capability_need_indicators = spark.createDataFrame(ref_rows, ["capability", "indicator", "direction", "label"])
capability_need_indicators.write.mode("overwrite").saveAsTable("workspace.default.capability_need_indicators")

print("district_capability_need built. Highest-need districts for Maternity / Delivery:")
(district_capability_need.filter(F.col("capability") == "Maternity / Delivery")
    .orderBy(F.desc("need_score")).select("dist_resolved", "need_score", "need_level").show(6, truncate=False))


/databricks/python/lib/python3.11/site-packages/pyspark/sql/connect/expressions.py:1017: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


district_capability_need built. Highest-need districts for Maternity / Delivery:
+--------------------+----------+----------+
|dist_resolved       |need_score|need_level|
+--------------------+----------+----------+
|Kishanganj          |97.0      |High      |
|Purnia              |96.0      |High      |
|Unakoti             |96.0      |High      |
|Deoghar             |95.0      |High      |
|Araria              |95.0      |High      |
|Pashchimi Singhbhum |94.0      |High      |
+--------------------+----------+----------+
only showing top 6 rows


In [0]:
# --- demand-supply gap: need vs verified (Strong) supply per district x capability ---
pin_lookup = (pin.withColumn("pincode6", F.regexp_extract(F.col("pincode").cast("string"), r"(\d{6})", 1))
                 .filter(F.col("pincode6") != "")
                 .groupBy("pincode6").agg(F.first("district").alias("dist_dir"))
                 .withColumn("dist_norm", _norm_d("dist_dir")))

# Strong-evidence supply per district x capability (evidence_summary carries pincode6 from §16)
supply = (evidence_summary.filter(F.col("evidence_grade") == "Strong")
            .join(pin_lookup, "pincode6", "left")
            .groupBy("dist_norm", "capability")
            .agg(F.countDistinct("unique_id").alias("strong_supply")))

district_capability_gap = (district_capability_need
    .join(supply, ["dist_norm", "capability"], "left")
    .withColumn("strong_supply", F.coalesce("strong_supply", F.lit(0)))
    .withColumn("gap_flag",
        F.when((F.col("need_level") == "High") & (F.col("strong_supply") == 0), "CRITICAL: high need, no verified supply")
         .when((F.col("need_level") == "High") & (F.col("strong_supply") <= 2), "PRIORITY: high need, thin supply")
         .otherwise("ok")))
district_capability_gap.createOrReplaceTempView("district_capability_gap")
district_capability_gap.write.mode("overwrite").saveAsTable("workspace.default.district_capability_gap")

print("biggest demand-supply gaps (high need, little/no verified supply):")
(district_capability_gap.filter(F.col("gap_flag") != "ok")
    .orderBy(F.desc("need_score"))
    .select("dist_resolved", "capability", "need_score", "strong_supply", "gap_flag")
    .show(12, truncate=55))

gaps = district_capability_gap.filter(F.col("gap_flag") != "ok").count()
readiness["need_aware_gap_districts"] = gaps
print(f"\ndistrict x capability gaps flagged for honest routing caveats: {gaps}")


/databricks/python/lib/python3.11/site-packages/pyspark/sql/connect/expressions.py:1017: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


biggest demand-supply gaps (high need, little/no verified supply):
+-------------------+------------------------+----------+-------------+---------------------------------------+
|      dist_resolved|              capability|need_score|strong_supply|                               gap_flag|
+-------------------+------------------------+----------+-------------+---------------------------------------+
|    Pathanamthitta |Endocrinology & Diabetes|     100.0|            0|CRITICAL: high need, no verified supply|
|            Nawada |       Oncology / Cancer|     100.0|            0|CRITICAL: high need, no verified supply|
|          Kottayam |Endocrinology & Diabetes|     100.0|            0|CRITICAL: high need, no verified supply|
|          Thrissur |Endocrinology & Diabetes|     100.0|            0|CRITICAL: high need, no verified supply|
|            Jalaun |       Oncology / Cancer|     100.0|            0|CRITICAL: high need, no verified supply|
|             Wokha |       Oncology 

## 19. Three-score readiness + referral-ready funnel

A single "trust" number conflates three different questions, and a facility can ace one
while failing another. This section splits readiness into three independent scores and
adds an honest **referral-ready funnel**.

- **Authenticity** (= §12 verification_score) — *is this a real, reachable facility?*
  Built from independent-source corroboration, named staff, logo, social, contact.
- **Clinical matchability** — *can we tell what care it actually provides?* Built from
  graded capability evidence (§16) — operational Strong evidence weighted highest — and
  whether it carries any controlled-vocabulary specialty (§15).
- **Geographic usability** — *can we place it well enough to route?* Valid India
  coordinates and a pincode that resolves to a district.

Keeping them separate exposes the failure mode the data hides: facilities that look
**trustworthy online but are clinically thin** (high authenticity, low matchability) —
exactly the rows that should be down-ranked or badged "verify before referral".

The **referral-ready funnel** then answers, honestly: how many facilities can *appear*,
how many can *show matching evidence* (the cited-text the rubric requires), and how many
have *strong operational proof* — so the app knows when to attach a "limited evidence"
caveat. Output: `facility_scorecard`.

In [0]:
# ---------- CLINICAL MATCHABILITY (from §16 evidence + §15 capabilities) ----------
match_ev = (evidence_summary.groupBy("unique_id")
    .agg(F.sum((F.col("evidence_grade") == "Strong").cast("int")).alias("strong_caps"),
         F.countDistinct("capability").alias("evidenced_caps")))
match_sp = (facility_capabilities.groupBy("unique_id")
    .agg(F.countDistinct("specialty").alias("n_specialties"),
         F.sum((F.col("confidence") == "Confirmed").cast("int")).alias("confirmed_specs")))

scored = (fac.select("unique_id", "name", "facilityTypeId",
                     "verification_score", "trust_tier",
                     "latitude", "longitude", "pincode6", "has_contact")
    .join(match_ev, "unique_id", "left")
    .join(match_sp, "unique_id", "left")
    .fillna(0, subset=["strong_caps", "evidenced_caps", "n_specialties", "confirmed_specs"]))

scored = scored.withColumn("has_spec", (F.col("n_specialties") > 0).cast("int"))
# 60 pts for operational Strong breadth (capped at 3), 40 pts for carrying any specialty
scored = scored.withColumn("matchability_score",
    F.round(F.least(F.col("strong_caps"), F.lit(3)) / 3 * 60 + F.col("has_spec") * 40, 0))

# ---------- GEOGRAPHIC USABILITY ----------
lat = F.col("latitude").cast("double"); lon = F.col("longitude").cast("double")
valid_coords = (lat.between(6.0, 37.5) & lon.between(68.0, 97.5)).cast("int")
pin_codes = (pin.withColumn("pincode6", F.regexp_extract(F.col("pincode").cast("string"), r"(\d{6})", 1))
                .filter(F.col("pincode6") != "").select("pincode6").distinct()
                .withColumn("_resolvable", F.lit(1)))
scored = (scored.join(pin_codes, "pincode6", "left")
                .withColumn("resolvable", F.coalesce(F.col("_resolvable"), F.lit(0)))
                .withColumn("geo_score", valid_coords * 50 + F.col("resolvable") * 50))

# ---------- assemble the three-score card ----------
facility_scorecard = (scored
    .withColumnRenamed("verification_score", "authenticity_score")
    .withColumnRenamed("trust_tier", "authenticity_tier")
    .withColumn("matchability_tier",
        F.when(F.col("matchability_score") >= 60, "High")
         .when(F.col("matchability_score") >= 30, "Medium").otherwise("Low"))
    .withColumn("geo_tier",
        F.when(F.col("geo_score") >= 80, "High")
         .when(F.col("geo_score") >= 40, "Medium").otherwise("Low"))
    .select("unique_id", "name", "facilityTypeId",
            "authenticity_score", "authenticity_tier",
            "matchability_score", "matchability_tier",
            "geo_score", "geo_tier",
            "strong_caps", "n_specialties", "has_contact", "resolvable"))
facility_scorecard.createOrReplaceTempView("facility_scorecard")
facility_scorecard.write.mode("overwrite").saveAsTable("workspace.default.facility_scorecard")

print("three-score distribution (% by tier):")
for dim in ["authenticity_tier", "matchability_tier", "geo_tier"]:
    print("  " + dim.replace("_tier", "") + ":")
    facility_scorecard.groupBy(dim).count().orderBy(F.desc("count")).show()


three-score distribution (% by tier):
  authenticity:
+-----------------+-----+
|authenticity_tier|count|
+-----------------+-----+
|             High| 7317|
|           Medium| 2463|
|              Low|  308|
+-----------------+-----+

  matchability:
+-----------------+-----+
|matchability_tier|count|
+-----------------+-----+
|             High| 5497|
|           Medium| 4473|
|              Low|  118|
+-----------------+-----+

  geo:
+--------+-----+
|geo_tier|count|
+--------+-----+
|    High| 9537|
|  Medium|  432|
|     Low|  119|
+--------+-----+



In [0]:
# ---------- the hidden failure mode: trustworthy online but clinically thin ----------
thin = facility_scorecard.filter(
    (F.col("authenticity_tier") == "High") & (F.col("matchability_tier") != "High"))
print(f"High authenticity but NOT high matchability: {thin.count()} "
      f"({thin.count()/n*100:.1f}%) -> down-rank / 'verify before referral' badge")
thin.select("name", "facilityTypeId", "authenticity_score", "matchability_score", "strong_caps").show(8, truncate=40)

# ---------- referral-ready funnel ----------
has_evidence = evidence_summary.select("unique_id").distinct().withColumn("_anyev", F.lit(1))
has_strong = (evidence_summary.filter(F.col("evidence_grade") == "Strong")
              .select("unique_id").distinct().withColumn("_strong", F.lit(1)))
funnel = (facility_scorecard
    .join(has_evidence, "unique_id", "left").join(has_strong, "unique_id", "left")
    .withColumn("geo_key", (F.col("geo_score") > 0).cast("int"))
    .withColumn("has_cap", ((F.col("n_specialties") > 0) | (F.coalesce("_anyev", F.lit(0)) == 1)).cast("int"))
    .withColumn("ready_min",
        ((F.col("name").isNotNull()) & (F.col("geo_key") == 1) &
         (F.col("has_contact") == 1) & (F.col("has_cap") == 1)).cast("int"))
    .withColumn("ready_evidence", ((F.col("ready_min") == 1) & (F.coalesce("_anyev", F.lit(0)) == 1)).cast("int"))
    .withColumn("ready_strong",   ((F.col("ready_min") == 1) & (F.coalesce("_strong", F.lit(0)) == 1)).cast("int")))

agg = funnel.agg(
    F.avg("ready_min").alias("min"), F.avg("ready_evidence").alias("ev"),
    F.avg("ready_strong").alias("strong")).collect()[0]
print("\nREFERRAL-READY FUNNEL:")
print(f"  meet minimum (can appear):        {agg['min']*100:5.1f}%")
print(f"  + can show matching evidence:     {agg['ev']*100:5.1f}%   <- the rubric's 'cite the text' bar")
print(f"  + has STRONG operational proof:   {agg['strong']*100:5.1f}%")
print(f"  -> {(agg['min']-agg['ev'])*100:.1f}% can appear but have NO citable evidence "
      f"(app must show a 'limited evidence' caveat)")

readiness["referral_ready_min_pct"] = round(agg["min"] * 100, 1)
readiness["referral_ready_evidence_pct"] = round(agg["ev"] * 100, 1)
readiness["clinically_thin_high_auth_pct"] = round(thin.count() / n * 100, 1)


High authenticity but NOT high matchability: 2723 (27.0%) -> down-rank / 'verify before referral' badge
+----------------------------------------+--------------+------------------+------------------+-----------+
|                                    name|facilityTypeId|authenticity_score|matchability_score|strong_caps|
+----------------------------------------+--------------+------------------+------------------+-----------+
|         MNR Dental College and Hospital|      hospital|             100.0|              40.0|          0|
|                      Tulsi Eye Hospital|      hospital|             100.0|              40.0|          0|
|                      Orbit Eye Hospital|      hospital|             100.0|              40.0|          0|
|                    Care Dental Hospital|      hospital|             100.0|              40.0|          0|
|Believers Church Medical College Hosp...|      hospital|             100.0|              40.0|          0|
|                     Ketkar Nur